# Question B：使用 Funnelling Approach 进行特征选择

本题使用沪深 300 指数数据构造较丰富的候选特征池，并通过 filter、wrapper 和 embedded 三类方法逐步筛选特征。候选特征覆盖收益、波动率、趋势、成交量、K 线结构和常用技术指标。

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import loguniform, randint, uniform
from xgboost import XGBClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score
from sklearn.utils.class_weight import compute_sample_weight

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

## 候选特征构造

In [ ]:
# Load file
df = pd.read_csv("CSI300_2005_2026.csv", parse_dates=["Date"]).sort_values("Date").set_index("Date")
df = df["2010":].copy()

# Core returns and price/volume structure
df["log_ret_1"] = np.log(df["Adj Close"]).diff()
df["simple_ret_1"] = df["Adj Close"].pct_change()
df["intraday_ret"] = df["Close"] / df["Open"] - 1
df["range_pct"] = df["High"] / df["Low"] - 1
df["gap_ret"] = df["Open"] / df["Close"].shift(1) - 1
df["upper_shadow"] = (df["High"] - df[["Open", "Close"]].max(axis=1)) / df["Close"]
df["lower_shadow"] = (df[["Open", "Close"]].min(axis=1) - df["Low"]) / df["Close"]
df["body_pct"] = (df["Close"] - df["Open"]) / df["Open"]
df["close_pos"] = (df["Close"] - df["Low"]) / (df["High"] - df["Low"]).replace(0, np.nan)
df["volume_ret"] = np.log(df["Volume"]).diff()
df["dow"] = df.index.dayofweek

# Rolling return, volatility, trend and volume features
for window in [2, 3, 5, 10, 20, 40, 60, 120]:
    df[f"ret_sum_{window}"] = df["log_ret_1"].rolling(window).sum()
    df[f"ret_mean_{window}"] = df["log_ret_1"].rolling(window).mean()
    df[f"volatility_{window}"] = df["log_ret_1"].rolling(window).std()
    df[f"ma_ratio_{window}"] = df["Adj Close"] / df["Adj Close"].rolling(window).mean() - 1
    df[f"volume_z_{window}"] = (
        df["Volume"] - df["Volume"].rolling(window).mean()
    ) / df["Volume"].rolling(window).std()
    df[f"rolling_min_ret_{window}"] = df["log_ret_1"].rolling(window).min()
    df[f"rolling_max_ret_{window}"] = df["log_ret_1"].rolling(window).max()

# RSI indicators
for window in [6, 14, 21]:
    delta = df["Adj Close"].diff()
    gain = delta.clip(lower=0).rolling(window).mean()
    loss = (-delta.clip(upper=0)).rolling(window).mean()
    rs = gain / loss.replace(0, np.nan)
    df[f"rsi_{window}"] = 100 - (100 / (1 + rs))

# MACD indicators
ema12 = df["Adj Close"].ewm(span=12, adjust=False).mean()
ema26 = df["Adj Close"].ewm(span=26, adjust=False).mean()
df["macd"] = ema12 - ema26
df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean()
df["macd_hist"] = df["macd"] - df["macd_signal"]

# ATR indicators
true_range = pd.concat(
    [
        df["High"] - df["Low"],
        (df["High"] - df["Close"].shift(1)).abs(),
        (df["Low"] - df["Close"].shift(1)).abs(),
    ],
    axis=1,
).max(axis=1)
for window in [14, 20]:
    df[f"atr_{window}"] = true_range.rolling(window).mean() / df["Adj Close"]

# Target: effective next-day uptrend
target_threshold = 0.0015
df["next_log_ret"] = df["log_ret_1"].shift(-1)
df["Label"] = np.where(df["next_log_ret"] > target_threshold, 1, 0)
df.loc[df["next_log_ret"].isna(), "Label"] = np.nan

excluded = ["Open", "High", "Low", "Close", "Adj Close", "Volume", "next_log_ret", "Label"]
features_list = [col for col in df.columns if col not in excluded]

data = df[features_list + ["Label", "next_log_ret"]].replace([np.inf, -np.inf], np.nan).dropna()
X = data[features_list]
y = data["Label"].astype(int).values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)

pd.DataFrame({
    "Item": ["Usable observations", "Training observations", "Testing observations", "Candidate features", "Training positive rate", "Testing positive rate"],
    "Value": [len(data), len(X_train), len(X_test), len(features_list), round(y_train.mean(), 4), round(y_test.mean(), 4)],
})

## Step 1：Filter 方法

Filter 阶段先删除训练集中相关系数绝对值高于 0.98 的冗余变量，然后用 mutual information 对剩余变量排序。

In [ ]:
corr = X_train.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
corr_dropped = [column for column in upper.columns if any(upper[column] > 0.98)]
corr_features = [column for column in X_train.columns if column not in corr_dropped]

mi_scores = pd.Series(
    mutual_info_classif(X_train[corr_features], y_train, random_state=42),
    index=corr_features,
).sort_values(ascending=False)

filter_features = list(mi_scores.head(min(64, len(mi_scores))).index)

print(f"Initial feature count: {len(features_list)}")
print(f"Dropped by correlation filter: {len(corr_dropped)}")
print(f"Features kept after filter step: {len(filter_features)}")
mi_scores.head(20).to_frame("mutual_information")

## Step 2：Wrapper 方法

Wrapper 阶段使用 XGBoost 和 `TimeSeriesSplit`，比较不同特征数量下的交叉验证 ROC AUC。由于标签存在轻微不平衡，训练中使用 `compute_sample_weight` 生成的样本权重。

In [ ]:
tscv = TimeSeriesSplit(n_splits=5, gap=1)
base_selector_params = {
    "verbosity": 0,
    "eval_metric": "logloss",
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1,
    "n_estimators": 120,
    "max_depth": 2,
    "learning_rate": 0.035,
    "subsample": 0.75,
    "colsample_bytree": 0.75,
    "min_child_weight": 5,
    "gamma": 1.0,
    "reg_alpha": 0.5,
    "reg_lambda": 3.0,
}

wrapper_rows = []
for n_features in [10, 15, 20, 25, 30, 35, 40, 50, len(filter_features)]:
    cols = filter_features[: min(n_features, len(filter_features))]
    selector_model = XGBClassifier(**base_selector_params)
    cv_scores = cross_val_score(
        selector_model,
        X_train[cols],
        y_train,
        cv=tscv,
        scoring="roc_auc",
        params={"sample_weight": sample_weights},
        n_jobs=1,
    )
    wrapper_rows.append({
        "n_features": len(cols),
        "cv_roc_auc_mean": cv_scores.mean(),
        "cv_roc_auc_std": cv_scores.std(),
        "features": cols,
    })

wrapper_table = pd.DataFrame(wrapper_rows).drop_duplicates("n_features")
best_wrapper = wrapper_table.sort_values(["cv_roc_auc_mean", "n_features"], ascending=[False, True]).iloc[0]
wrapper_features = list(best_wrapper["features"])

wrapper_table.drop(columns=["features"]).round(4)

## Step 3：Embedded 方法

Embedded 阶段在 wrapper 选出的特征上训练 XGBoost，并使用 gain importance 排序；随后再次用时间序列交叉验证决定最终保留数量。

In [ ]:
embedded_model = XGBClassifier(**base_selector_params, importance_type="gain")
embedded_model.fit(X_train[wrapper_features], y_train, sample_weight=sample_weights)

gain_scores = pd.Series(
    embedded_model.feature_importances_,
    index=wrapper_features,
).sort_values(ascending=False)

embedded_rows = []
for n_features in [10, 15, 20, 25, 30, 35, 40, len(gain_scores)]:
    cols = list(gain_scores.head(min(n_features, len(gain_scores))).index)
    selector_model = XGBClassifier(**base_selector_params)
    cv_scores = cross_val_score(
        selector_model,
        X_train[cols],
        y_train,
        cv=tscv,
        scoring="roc_auc",
        params={"sample_weight": sample_weights},
        n_jobs=1,
    )
    embedded_rows.append({
        "n_features": len(cols),
        "cv_roc_auc_mean": cv_scores.mean(),
        "cv_roc_auc_std": cv_scores.std(),
        "features": cols,
    })

embedded_table = pd.DataFrame(embedded_rows).drop_duplicates("n_features")
best_embedded = embedded_table.sort_values(["cv_roc_auc_mean", "n_features"], ascending=[False, True]).iloc[0]
final_features = list(best_embedded["features"])

display(embedded_table.drop(columns=["features"]).round(4))

pd.DataFrame({
    "Rank": range(1, len(final_features) + 1),
    "Feature": final_features,
    "Gain": gain_scores.loc[final_features].round(6).values,
})

## 特征选择结论

最终特征由三层漏斗共同决定：相关性和互信息完成初筛，时间序列交叉验证完成 wrapper 选择，XGBoost gain importance 和再次交叉验证完成 embedded 选择。该特征子集将用于第 3 题的模型训练与调参。